# 4.3 各位置的數值尺度怎麼控制？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：每張卡片自己調尺度**

沿 feature 軸，不能跨時間偷看別的卡片。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/normalization.svg")))

**先想一想：**把別筆資料放進batch，原本token的LN會改變嗎？

每一個 token 自己計算 feature 平均和變異數，避免數值尺度漂移。不要跨時間平均，否則會混入其他 token，甚至未來資訊。

**把直覺寫成數學：**$LN(x)=(x-\mu)/\sqrt{var(x)+\epsilon}\cdot\gamma+\beta$；μ/var 沿最後 feature 軸。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
x = torch.tensor([[[1.0, 2.0, 3.0], [10.0, 20.0, 30.0]]])
layer = nn.LayerNorm(3)
y = layer(x)
print(y, y.mean(-1), y.var(-1, unbiased=False))
assert torch.allclose(y.mean(-1), torch.zeros(1, 2), atol=1e-6)

**如何讀結果：**epsilon 避免零variance；小epsilon下variance接近1，不必硬等於1。

**只改一件事：**只在第二token加100，檢查第一token不變。
